# Parts-of-Speech Tagging - Working with tags and Numpy

在本节讲座笔记中，你将使用一些词性标注信息创建一个矩阵，然后使用不同的方法对其进行修改。
这将作为使用 Numpy 的动手实践体验，同时也是对用于词性标注（POS tagging）的一些元素的介绍。

In [18]:
import numpy as np
import pandas as pd

### Some information on tags

For this notebook you will be using a toy example including only three tags (or states). In a real world application there are many more tags which can be found [here](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html).

In [19]:
# Define tags for Adverb, Noun and To (the preposition) , respectively
tags = ['RB', 'NN', 'TO']

在本周的作业中，你将构建一些字典，这些字典提供有关你将使用的词性标签和单词的有用信息。

其中一个是 `transition_counts` 字典，它统计某个特定标签出现在另一个标签旁边的次数。该字典的键形式为 `(previous_tag, tag)`，其值为出现频率。

另一个是 `emission_counts` 字典，它将统计训练数据集中特定 `(tag, word)` 对出现的次数。

通常，在处理仅涉及标签的情况时，请考虑使用 `transition`；而在处理标签和单词时，请考虑使用 `emission`。

在本笔记本中，你将查看第一个字典：

In [20]:
# Define 'transition_counts' dictionary
# Note: values are the same as the ones in the assignment
transition_counts = {
    ('NN', 'NN'): 16241,
    ('RB', 'RB'): 2263,
    ('TO', 'TO'): 2,
    ('NN', 'TO'): 5256,
    ('RB', 'TO'): 855,
    ('TO', 'NN'): 734,
    ('NN', 'RB'): 2431,
    ('RB', 'NN'): 358,
    ('TO', 'RB'): 200
}

Notice that there are 9 combinations of the 3 tags used. Each tag can appear after the same tag so you should include those as well.

### Using Numpy for matrix creation

Now you will create a matrix that includes these frequencies using Numpy arrays:

In [21]:
# Store the number of tags in the 'num_tags' variable
num_tags = len(tags)

# Initialize a 3X3 numpy array with zeros
transition_matrix = np.zeros((num_tags, num_tags))

# Print matrix
transition_matrix

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

Visually you can see the matrix has the correct dimensions. Don't forget you can check this too using the `shape` attribute:

In [22]:
# Print shape of the matrix
transition_matrix.shape

(3, 3)

Before filling this matrix with the values of the `transition_counts` dictionary you should sort the tags so that their placement in the matrix is consistent:

In [23]:
# Create sorted version of the tag's list
sorted_tags = sorted(tags)

# Print sorted list
sorted_tags

['NN', 'RB', 'TO']

To fill this matrix with the correct values you can use a `double for loop`. You could also use `itertools.product` to one line this double loop:

In [24]:
# Loop rows
for i in range(num_tags):
    # Loop columns
    for j in range(num_tags):
        # Define tag pair
        tag_tuple = (sorted_tags[i], sorted_tags[j])
        # Get frequency from transition_counts dict and assign to (i, j) position in the matrix
        transition_matrix[i, j] = transition_counts.get(tag_tuple)

# Print matrix
transition_matrix

array([[1.6241e+04, 2.4310e+03, 5.2560e+03],
       [3.5800e+02, 2.2630e+03, 8.5500e+02],
       [7.3400e+02, 2.0000e+02, 2.0000e+00]])

看起来这运行良好。然而，该矩阵可能难以阅读，因为 `Numpy` 更注重效率，而不是以漂亮的格式呈现数值。

为此，你可以使用 `Pandas DataFrame`。特别地，一个以矩阵为输入并打印出其美观版本的函数将非常有用：

In [25]:
# Define 'print_matrix' function
def print_matrix(matrix):
    print(pd.DataFrame(matrix, index=sorted_tags, columns=sorted_tags))

Notice that the tags are not a parameter of the function. This is because the `sorted_tags` list will not change in the rest of the notebook so it is safe to use the variable previously declared. To test this function simply run: 

In [26]:
# Print the 'transition_matrix' by calling the 'print_matrix' function
print_matrix(transition_matrix)

         NN      RB      TO
NN  16241.0  2431.0  5256.0
RB    358.0  2263.0   855.0
TO    734.0   200.0     2.0


这样好多了，不是吗？

你可能已经推断出，这个矩阵是非对称的。

### Working with Numpy for matrix manipulation

现在你已经设置好了矩阵，接下来该看看矩阵在创建之后如何被操作了。

`Numpy` 支持向量化操作，这意味着通常需要遍历矩阵的操作可以以更简单的方式完成。这与将 numpy 数组视为矩阵的做法一致，因为你可以获得对常见矩阵操作的支持。你可以进行矩阵乘法、标量乘法、向量加法等等！

例如，尝试将矩阵中的每个值按 $\frac{1}{10}$ 的因子进行缩放。通常你会遍历矩阵中的每个值并相应地更新它们。但在 Numpy 中，这就像将整个矩阵除以 10 一样简单：

In [27]:
# Scale transition matrix
transition_matrix = transition_matrix/10

# Print scaled matrix
print_matrix(transition_matrix)

        NN     RB     TO
NN  1624.1  243.1  525.6
RB    35.8  226.3   85.5
TO    73.4   20.0    0.2


另一个更棘手的例子是归一化每一行，使得每个值等于 $\frac{value}{行的总和}$。

这可以通过向量化轻松完成。首先，你将计算每一行的总和：

In [28]:
# Compute sum of row for each row
rows_sum = transition_matrix.sum(axis=1, keepdims=True)

# Print sum of rows
rows_sum

array([[2392.8],
       [ 347.6],
       [  93.6]])

请注意，这里使用了 `sum()` 方法。该方法的功能正如其名称所示。由于需要的是行的总和，因此将 `axis` 设置为 `1`。在 Numpy 中，`axis=1` 指的是列，因此求和是通过对特定行的每一列求和来完成的，针对每一行。

另外，`keepdims` 参数被设置为 `True`，这样得到的数组形状为 `(3, 1)` 而不是 `(3,)`。这样做是为了使轴与所需的操作保持一致。

在使用 Numpy 时，请务必记住检查你所处理的数组的形状，许多意外的错误都是由于轴不一致导致的。在这些情况下，`shape` 属性是你的好帮手。

In [29]:
# Normalize transition matrix
transition_matrix = transition_matrix / rows_sum

# Print normalized matrix
print_matrix(transition_matrix)

          NN        RB        TO
NN  0.678745  0.101596  0.219659
RB  0.102992  0.651036  0.245972
TO  0.784188  0.213675  0.002137


请注意，所执行的归一化操作强制使每一行的总和等于 `1`。你可以通过对结果矩阵运行 `sum` 方法轻松地验证这一点：

In [30]:
transition_matrix.sum(axis=1, keepdims=True)

array([[1.],
       [1.],
       [1.]])

在最后一个示例中，你被要求修改矩阵对角线上的每个值，使其等于当前行总和加上当前值之和的对数（log）。在进行此类数学运算时，别忘了导入 `math` 模块。

这可以使用标准的 `for 循环` 或 `向量化` 来完成。你将看到这两种方式的实际应用：

In [31]:
import math

# Copy transition matrix for for-loop example
t_matrix_for = np.copy(transition_matrix)

# Copy transition matrix for numpy functions example
t_matrix_np = np.copy(transition_matrix)

#### Using a for-loop

In [32]:
# Loop values in the diagonal
for i in range(num_tags):
    t_matrix_for[i, i] = t_matrix_for[i, i] + math.log(rows_sum[i, 0])

# Print matrix
print_matrix(t_matrix_for)

          NN        RB        TO
NN  8.458964  0.101596  0.219659
RB  0.102992  6.502088  0.245972
TO  0.784188  0.213675  4.541167


#### Using vectorization

In [33]:
# Save diagonal in a numpy array
d = np.diag(t_matrix_np)

# Print shape of diagonal
d.shape

(3,)

You can save the diagonal in a numpy array using Numpy's `diag()` function. Notice that this array has shape `(3,)` so it is inconsistent with the dimensions of the `rows_sum` array which are `(3, 1)`. You'll have to reshape before moving forward. For this you can use Numpy's `reshape()` function, specifying the desired shape in a tuple:

In [34]:
# Reshape diagonal numpy array
d = np.reshape(d, (3,1))

# Print shape of diagonal
d.shape

(3, 1)

现在对角线已经具有正确的形状，你可以通过对 `rows_sum` 数组应用 `math.log()` 函数并加上对角线值来执行向量化操作。

要对 numpy 数组的每个元素应用一个函数，请使用 Numpy 的 `vectorize()` 函数，并将所需函数作为参数提供。该函数返回一个向量化函数，该函数接受一个 numpy 数组作为参数。

要更新原始矩阵，你可以使用 Numpy 的 `fill_diagonal()` 函数。

In [35]:
# Perform the vectorized operation
d = d + np.vectorize(math.log)(rows_sum)

# Use numpy's 'fill_diagonal' function to update the diagonal
np.fill_diagonal(t_matrix_np, d)

# Print the matrix
print_matrix(t_matrix_np)

          NN        RB        TO
NN  8.458964  0.101596  0.219659
RB  0.102992  6.502088  0.245972
TO  0.784188  0.213675  4.541167


To perform a sanity check that both methods yield the same result you can compare both matrices. Notice that this operation is also vectorized so you will get the equality check for each element in both matrices:

In [36]:
# Check for equality
t_matrix_for == t_matrix_np

array([[ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True]])

**Congratulations on finishing this lecture notebook!** Now you should be more familiar with some elements used by a POS tagger such as the `transition_counts` dictionary and with working with Numpy.

**Keep it up!**

In [37]:
import math
import numpy as np

rows_sum = np.array([[2392.8], [347.6], [93.6]])
t_matrix_for = np.array([[0.678745, 0.101596, 0.219659], [0.102992, 0.651036, 0.245972], [0.784188, 0.213675, 0.002137]])
num_tags = 3
for i in range(num_tags):
    t_matrix_for[i, i] = t_matrix_for[i, i] + math.log(rows_sum[i, 0])

assert t_matrix_for.shape == (3, 3)
assert np.isfinite(t_matrix_for).all()
t_matrix_for

array([[8.45896451, 0.101596  , 0.219659  ],
       [0.102992  , 6.50208839, 0.245972  ],
       [0.784188  , 0.213675  , 4.54116738]])